<a href="https://colab.research.google.com/github/Kangai100/Bol/blob/main/Netflix_movie_recommendation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **NETFLIX MOVIE and TV SHOW RECOMMENDATION SYSTEM**

## Importing the libraries

In [1]:
import pandas as pd
import numpy as np

## Loading the dataset from kaggle

In [2]:
import kagglehub
path = kagglehub.dataset_download("shivamb/netflix-shows")

Using Colab cache for faster access to the 'netflix-shows' dataset.


In [3]:
import os
path = kagglehub.dataset_download("shivamb/netflix-shows")

csv_path = os.path.join(path, "netflix_titles.csv")

df = pd.read_csv(csv_path)

df.head(3)

Using Colab cache for faster access to the 'netflix-shows' dataset.


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...


In [4]:
df.tail(2)

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
8805,s8806,Movie,Zoom,Peter Hewitt,"Tim Allen, Courteney Cox, Chevy Chase, Kate Ma...",United States,"January 11, 2020",2006,PG,88 min,"Children & Family Movies, Comedies","Dragged from civilian life, a former superhero..."
8806,s8807,Movie,Zubaan,Mozez Singh,"Vicky Kaushal, Sarah-Jane Dias, Raaghav Chanan...",India,"March 2, 2019",2015,TV-14,111 min,"Dramas, International Movies, Music & Musicals",A scrappy but poor boy worms his way into a ty...


## checking for missing values

In [5]:
df.isnull().sum()

,0
show_id,0
type,0
title,0
director,2634
cast,825
country,831
date_added,10
release_year,0
rating,4
duration,3


## filling in the text rows columns with an empty string

In [6]:
# the empty rows are director, cast, country
text_columns = ['director', 'cast', 'country']
for col in text_columns:
  df[col] = df[col].fillna('')

df.isnull().sum()

,0
show_id,0
type,0
title,0
director,0
cast,0
country,0
date_added,10
release_year,0
rating,4
duration,3


## dropping the rows that have less missing values

In [7]:
df = df.dropna(subset = ['date_added', 'rating', 'duration'])
df.isnull().sum()

,0
show_id,0
type,0
title,0
director,0
cast,0
country,0
date_added,0
release_year,0
rating,0
duration,0


## i want the type column to only show movies not tv shows

In [8]:
movies_df = df[df['type'] == 'Movie'].reset_index(drop = True)
movies_df.head(3)

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s7,Movie,My Little Pony: A New Generation,"Robert Cullen, José Luis Ucha","Vanessa Hudgens, Kimiko Glenn, James Marsden, ...",,"September 24, 2021",2021,PG,91 min,Children & Family Movies,Equestria's divided. But a bright-eyed hero be...
2,s8,Movie,Sankofa,Haile Gerima,"Kofi Ghanaba, Oyafunmike Ogunlano, Alexandra D...","United States, Ghana, Burkina Faso, United Kin...","September 24, 2021",1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies","On a photo shoot in Ghana, an American model s..."


In [9]:
movies_df.isnull().sum()

,0
show_id,0
type,0
title,0
director,0
cast,0
country,0
date_added,0
release_year,0
rating,0
duration,0


## Combining text columns director, cast, country, listed_in,	description

In [10]:
def combine_features(row):
  return row['director'] + '' + row['cast'] + '' +row['country'] + '' + row['listed_in'] + '' + row['description']
movies_df['combine_features'] = movies_df.apply(combine_features, axis = 1)

movies_df.head(2)

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,combine_features
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm...",Kirsten JohnsonUnited StatesDocumentariesAs he...
1,s7,Movie,My Little Pony: A New Generation,"Robert Cullen, José Luis Ucha","Vanessa Hudgens, Kimiko Glenn, James Marsden, ...",,"September 24, 2021",2021,PG,91 min,Children & Family Movies,Equestria's divided. But a bright-eyed hero be...,"Robert Cullen, José Luis UchaVanessa Hudgens, ..."


## feature scalling using tfidf

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies_df['combine_features'].fillna(''))
tfidf_matrix.shape

(6126, 49041)

## Getting the similarities between words using the cosine

In [12]:
from sklearn.metrics.pairwise import cosine_similarity
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
cosine_sim

array([[1.        , 0.        , 0.        , ..., 0.        , 0.00932275,
        0.        ],
       [0.        , 1.        , 0.        , ..., 0.        , 0.01098779,
        0.02064928],
       [0.        , 0.        , 1.        , ..., 0.        , 0.00208089,
        0.01520838],
       ...,
       [0.        , 0.        , 0.        , ..., 1.        , 0.03700969,
        0.        ],
       [0.00932275, 0.01098779, 0.00208089, ..., 0.03700969, 1.        ,
        0.00719353],
       [0.        , 0.02064928, 0.01520838, ..., 0.        , 0.00719353,
        1.        ]])

## creating the recommendation system

In [13]:
indices = pd.Series(movies_df.index, index=movies_df['title'].str.lower()).drop_duplicates()

In [14]:
movies_df[movies_df['title'].str.lower().str.contains('avatar', na=False)][['title']]

,title


In [15]:
print(indices[indices.index.str.contains('The App', na=False)])

Series([], dtype: int64)


In [16]:
print(movies_df['title'].head(20))

0                                  Dick Johnson Is Dead
1                      My Little Pony: A New Generation
2                                               Sankofa
3                                          The Starling
4                                          Je Suis Karl
5                      Confessions of an Invisible Girl
6     Europe's Most Dangerous Man: Otto Skorzeny in ...
7                                             Intrusion
8                                       Avvai Shanmughi
9          Go! Go! Cory Carson: Chrissy Takes the Wheel
10                                                Jeans
11                                       Minsara Kanavu
12                                            Grown Ups
13                                           Dark Skies
14                                             Paranoia
15                                      Ankahi Kahaniya
16                       The Father Who Moves Mountains
17                                       The Str

In [17]:
def recommend_movies(title, cosine_sim = cosine_sim):
  idx = indices[title.lower()]

  sim_scores = list(enumerate(cosine_sim[idx]))

  sim_scores = sorted(sim_scores, key = lambda x: x[1], reverse = True)

  sim_scores = sim_scores[1:11]

  movie_indices = [i[0] for i in sim_scores]

  return movies_df['title'].iloc[movie_indices]
choice = recommend_movies("Jaws")

print("Movies recommended for Jaws:")
for movie in choice:
    print(movie)

Movies recommended for Jaws:
Jaws 2
Jaws: The Revenge
Jaws 3
The Laundromat
In The Deep
Soul Surfer
Iliza Shlesinger: Confirmed Kills
Paranoia
Mission Blue
Monster Hunter: Legends of the Guild


# Tv Shows dataset

In [18]:
TvShow_df = df[df['type'] == 'TV Show'].reset_index(drop = True)
TvShow_df.head(3)

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s2,TV Show,Blood & Water,,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
1,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
2,s4,TV Show,Jailbirds New Orleans,,,,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."


## combining text columns for TvShows dataset

In [19]:
def combine_features1(row):
  return row['director'] + '' + row['cast'] + '' + row['country'] + '' + row['listed_in'] + '' + row['description']
TvShow_df['combine_features1'] = TvShow_df.apply(combine_features1, axis = 1)

TvShow_df.head(2)

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,combine_features1
0,s2,TV Show,Blood & Water,,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t...","Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban..."
1,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...,"Julien LeclercqSami Bouajila, Tracy Gotoas, Sa..."


In [23]:
TvShow_df.tail(3)

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,combine_features1
2661,s8798,TV Show,Zak Storm,,"Michael Johnston, Jessica Gee-George, Christin...","United States, France, South Korea, Indonesia","September 13, 2018",2016,TV-Y7,3 Seasons,Kids' TV,Teen surfer Zak Storm is mysteriously transpor...,"Michael Johnston, Jessica Gee-George, Christin..."
2662,s8801,TV Show,Zindagi Gulzar Hai,,"Sanam Saeed, Fawad Khan, Ayesha Omer, Mehreen ...",Pakistan,"December 15, 2016",2012,TV-PG,1 Season,"International TV Shows, Romantic TV Shows, TV ...","Strong-willed, middle-class Kashaf and carefre...","Sanam Saeed, Fawad Khan, Ayesha Omer, Mehreen ..."
2663,s8804,TV Show,Zombie Dumb,,,,"July 1, 2019",2018,TV-Y7,2 Seasons,"Kids' TV, Korean TV Shows, TV Comedies","While living alone in a spooky town, a young g...","Kids' TV, Korean TV Shows, TV ComediesWhile li..."


## feature scaling of TvShow_df using the tfidfvectorizer

In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf1 = TfidfVectorizer(stop_words='english')
tfidf_matrix1 = tfidf1.fit_transform(TvShow_df['combine_features1'].fillna(''))
tfidf_matrix1.shape

(2664, 26735)

## Getting the similarities using the cosine similarity

In [21]:
from sklearn.metrics.pairwise import cosine_similarity
cosine_sim1 = cosine_similarity(tfidf_matrix1, tfidf_matrix1)
cosine_sim1

array([[1.        , 0.00765568, 0.        , ..., 0.        , 0.00790404,
        0.02998392],
       [0.00765568, 1.        , 0.        , ..., 0.        , 0.01254006,
        0.01859919],
       [0.        , 0.        , 1.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 1.        , 0.        ,
        0.        ],
       [0.00790404, 0.01254006, 0.        , ..., 0.        , 1.        ,
        0.01920258],
       [0.02998392, 0.01859919, 0.        , ..., 0.        , 0.01920258,
        1.        ]])

## creating the recommendation system

### first is to get the indices of the rows

In [22]:
indices1 = pd.Series(TvShow_df.index, index=TvShow_df['title'].str.lower()).drop_duplicates()

In [24]:
def recommend_TvShow(title, cosine_sim1 = cosine_sim1):

  idx1 = indices1[title.lower()]

  sim_scores1 = list(enumerate(cosine_sim1[idx1]))

  sim_scores1 = sorted(sim_scores1, key = lambda x: x[1], reverse = True)

  sim_scores1 = sim_scores1[1:11]

  TvShow_indices = [i[0] for i in sim_scores1]

  return  TvShow_df['title'].iloc[TvShow_indices]
choice1 = recommend_TvShow("Zindagi Gulzar Hai")

print("Tvshow recommended for Zindagi Gulzar Hai:")
for Tv_Show in choice1:
    print(Tv_Show)

Tvshow recommended for Zindagi Gulzar Hai:
Humsafar
Khaani
Love Me As I Am
Sadqay Tumhare
Bangkok Bachelors
Yeh Meri Family
My Husband Won't Fit
Revolutionary Love
Cinta 100KG
Encounters with Evil
